# 06 — Inference, market ranking, and allocation
Convert predictions into a transparent decision while keeping transport cost, risk penalty, and capacity explicit.

In [ ]:
%pip install -q -e .
import json
import pandas as pd
from agridecision.decision.ranking import rank_markets
from agridecision.decision.optimization import optimise_market_allocation
from agridecision.models.bundle import ModelBundle

In [ ]:
features = pd.read_csv('data/processed/supervised_features.csv', parse_dates=['arrival_date'])
contract = json.loads(open('artifacts/feature_contract.json').read())
latest = features.sort_values('arrival_date').groupby('market', as_index=False).tail(1)
predicted = ModelBundle('artifacts').predict(latest)
predicted[['market', 'predicted_price', 'prediction_lower_90', 'prediction_upper_90', 'shock_probability', 'anomaly_score']]

Provide real road distance or route distance before using the ranking operationally. The values here are demonstration inputs.

In [ ]:
demo_distance = {market: 25 + index * 35 for index, market in enumerate(sorted(predicted['market'].unique()))}
decision_input = predicted.assign(
    distance_km=predicted['market'].map(demo_distance),
    capacity=100.0,
)
ranked = rank_markets(decision_input, transport_cost_per_km_quintal=1.2, risk_penalty=250)
ranked[['recommendation_rank', 'market', 'predicted_price', 'transport_cost', 'shock_probability', 'expected_net_price', 'decision_utility']]

In [ ]:
allocation = optimise_market_allocation(ranked, available_quantity=200, maximum_weight_per_market=0.4)
print('Optimisation success:', allocation.success)
print('Expected risk-adjusted total value:', allocation.expected_total_value)
allocation.allocations